# OsteoInsight — BTXRD lesion-localization workflow

This notebook audits the original BTXRD release, creates leakage-aware YOLO manifests without modifying source files, trains a single-class lesion detector, and evaluates the untouched test split once. Bounding boxes reproduce dataset annotations; they do not establish malignancy or clinical diagnosis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!python -m pip install -q 'ultralytics>=8.3,<9' 'pandas>=2.1,<3' 'Pillow>=10.2,<13' 'openpyxl>=3.1,<4' 'opencv-python-headless>=4.9,<5'

## Set the two Drive paths
`BTXRD_ROOT` must contain `images/`, `Annotations/`, and `dataset.csv` or `dataset.xlsx`. The source directory is read only.

In [ ]:
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/OsteoInsight-AI')
BTXRD_ROOT = Path('/content/drive/MyDrive/BTXRD')
assert (PROJECT / 'btxrd_prepare.py').is_file(), f'Project scripts not found: {PROJECT}'
assert (BTXRD_ROOT / 'images').is_dir(), f'images/ missing under {BTXRD_ROOT}'
assert (BTXRD_ROOT / 'Annotations').is_dir(), f'Annotations/ missing under {BTXRD_ROOT}'
assert any((BTXRD_ROOT / name).is_file() for name in ('dataset.csv', 'dataset.xlsx')), 'dataset.csv/xlsx missing'
%cd {PROJECT}

In [ ]:
# Audit first. This deliberately stops on blocking annotation/data issues.
!python btxrd_prepare.py "{BTXRD_ROOT}" --output-dir data/btxrd_detector --report-dir reports/btxrd_audit --seed 42

import json
audit = json.loads((PROJECT / 'reports/btxrd_audit/audit_summary.json').read_text())
audit

In [ ]:
# Train only after the audit cell succeeds and its reports have been reviewed.
import torch
device = '0' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Training device:', device)
!python train_detector.py data/btxrd_detector/data.yaml --weights yolov8n.pt --epochs 100 --patience 20 --image-size 640 --batch-size 16 --device "{device}" --seed 42 --output models/btxrd_yolov8n_best.pt

In [ ]:
# Final evaluation: run once after model selection is finished.
!python evaluate_detector.py data/btxrd_detector/manifest.csv --data data/btxrd_detector/data.yaml --checkpoint models/btxrd_yolov8n_best.pt --output-dir reports/btxrd_evaluation --confidence 0.25 --iou 0.5 --device "{device}"

metrics = json.loads((PROJECT / 'reports/btxrd_evaluation/metrics.json').read_text())
metrics

In [ ]:
# Reproducibility and handoff evidence.
import hashlib
checkpoint = PROJECT / 'models/btxrd_yolov8n_best.pt'
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
print('Checkpoint:', checkpoint)
print('SHA-256:', digest)
print('mAP@0.5:', metrics['map_0_5'])
print('mAP@0.5:0.95:', metrics['map_0_5_0_95'])
print('Lesion recall:', metrics['lesion_detection_recall'])
print('Image sensitivity:', metrics['image_sensitivity'])
print('Image specificity:', metrics['image_specificity'])

## Application handoff
Copy `models/btxrd_yolov8n_best.pt`, `models/btxrd_training_config.json`, and `reports/btxrd_evaluation/` back to the same project directory on the Mac. FastAPI will load that exact checkpoint on its next restart. Keep the declared application threshold aligned with the evaluated threshold.